# Transcript/Site-Level Benchmarking

Compares per-**position** (site-level) modification scores across **all baleen pipeline stages**.

## Methods Compared

| Label | Source | Score Definition |
|-------|--------|-----------------|
| `V1 z-score` | hierarchical V1 | Mean native z-score under empirical-Bayes null |
| `V2 raw` | `p_mod_raw` | Mean native V2 anchored mixture posterior |
| `V2 kNN` | `p_mod_knn` | Mean native kNN IVT-purity score |
| `V3 HMM` | `p_mod_hmm` | Mean native HMM-smoothed P(modified) |
| `Stoichiometry` | V3 HMM | Fraction of native reads with P(mod) > 0.5 |
| `-log10(padj)` | site aggregation | BH-adjusted p-value from Mann-Whitney U |

### Stoichiometry

**Stoichiometry** (also called "modification ratio" or "mod fraction") estimates **what proportion of RNA molecules are modified at a given position**. Many RNA modifications are *sub-stoichiometric*: not every transcript copy carries the modification. For example, a stoichiometry of 0.6 means ~60 % of sequenced native reads show the modification signal at that position.

In baleen, stoichiometry is computed as the fraction of native reads whose HMM-smoothed posterior P(modified) exceeds 0.5:

$$\text{stoichiometry} = \frac{1}{N_{\text{native}}} \sum_{i=1}^{N_{\text{native}}} \mathbb{1}\!\left[P_{\text{HMM}}^{(i)} > 0.5\right]$$

This is a hard-threshold estimate. A complementary soft estimate (`mean_p_mod`) averages the posteriors directly without thresholding. Both are reported in `site_results.tsv`.

## 1. Imports and Configuration

In [ ]:
import sys
from pathlib import Path

if Path.cwd().name == 'notebooks':
    sys.path.insert(0, str(Path.cwd().parent))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from collections import Counter
from scipy import stats
from sklearn.metrics import (
    roc_curve, auc, precision_recall_curve,
    average_precision_score, f1_score,
)
from sklearn.calibration import calibration_curve

# Baleen imports
from baleen.eventalign import load_results
from baleen.eventalign._hierarchical import compute_sequential_modification_probabilities
from baleen.eventalign._aggregation import aggregate_all, write_site_tsv

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 130, 'font.size': 11})

print('Imports OK')

## 2. File Paths — Edit These

In [ ]:
# ── Baleen outputs ───────────────────────────────────────────────────
BALEEN_PKL   = Path('../output/pipeline_results.pkl')
BALEEN_SITES = Path('../output/site_results.tsv')   # optional, can be regenerated

# ── Ground-truth offset ──────────────────────────────────────────────
POSITION_OFFSET = 0  # pipeline now emits 1-based center-of-kmer positions

# ── Output ───────────────────────────────────────────────────────────
OUT_DIR = Path('figures_transcript_level')
OUT_DIR.mkdir(exist_ok=True)

print('Configuration set.')

## 3. Ground Truth: Known Modification Sites (E. coli rRNA)

In [ ]:
KNOWN_MODIFICATIONS = {
    # Pseudouridine (Ψ)
    ('ecoli16S', 516):  ('Ψ',     'pseudouridine'),
    ('ecoli23S', 746):  ('Ψ',     'pseudouridine'),
    ('ecoli23S', 955):  ('Ψ',     'pseudouridine'),
    ('ecoli23S', 1911): ('Ψ',     'pseudouridine'),
    ('ecoli23S', 1917): ('Ψ',     'pseudouridine'),
    ('ecoli23S', 2457): ('Ψ',     'pseudouridine'),
    ('ecoli23S', 2504): ('Ψ',     'pseudouridine'),
    ('ecoli23S', 2580): ('Ψ',     'pseudouridine'),
    ('ecoli23S', 2604): ('Ψ',     'pseudouridine'),
    ('ecoli23S', 2605): ('Ψ',     'pseudouridine'),
    # N2-methylguanosine (m2G)
    ('ecoli16S', 966):  ('m2G',   'N2-methylguanosine'),
    ('ecoli16S', 1207): ('m2G',   'N2-methylguanosine'),
    ('ecoli16S', 1516): ('m2G',   'N2-methylguanosine'),
    ('ecoli23S', 1835): ('m2G',   'N2-methylguanosine'),
    ('ecoli23S', 2445): ('m2G',   'N2-methylguanosine'),
    # 5-methylcytidine (m5C)
    ('ecoli16S', 967):  ('m5C',   '5-methylcytidine'),
    ('ecoli16S', 1407): ('m5C',   '5-methylcytidine'),
    ('ecoli23S', 1962): ('m5C',   '5-methylcytidine'),
    # 5-methyluridine (m5U / ribothymidine)
    ('ecoli23S', 747):  ('m5U',   '5-methyluridine'),
    ('ecoli23S', 1939): ('m5U',   '5-methyluridine'),
    # N6,N6-dimethyladenosine (m6,6A)
    ('ecoli16S', 1518): ('m6,6A', 'N6,N6-dimethyladenosine'),
    ('ecoli16S', 1519): ('m6,6A', 'N6,N6-dimethyladenosine'),
    # N6-methyladenosine (m6A)
    ('ecoli23S', 1618): ('m6A',   'N6-methyladenosine'),
    ('ecoli23S', 2030): ('m6A',   'N6-methyladenosine'),
    # N7-methylguanosine (m7G)
    ('ecoli16S', 527):  ('m7G',   'N7-methylguanosine'),
    ('ecoli23S', 2069): ('m7G',   'N7-methylguanosine'),
    # Other modifications
    ('ecoli23S', 2498): ('Cm',    "2'-O-methylcytosine"),
    ('ecoli23S', 2449): ('D',     'dihydrouridine'),
    ('ecoli23S', 2251): ('Gm',    "2'-O-methylguanosine"),
    ('ecoli23S', 2552): ('Um',    "2'-O-methyluridine"),
    ('ecoli23S', 2501): ('ho5C',  '5-hydroxycytidine'),
    ('ecoli23S', 745):  ('m1G',   '1-methylguanosine'),
    ('ecoli23S', 2503): ('m2A',   '2-methyladenosine'),
    ('ecoli23S', 1915): ('m3Ψ',   '3-methylpseudouridine'),
    ('ecoli16S', 1498): ('m3U',   '3-methyluridine'),
    ('ecoli16S', 1402): ('m4Cm',  "N4,2'-O-dimethylcytidine"),
}

KNOWN_MOD_PIPELINE = {
    (contig, bio_pos - POSITION_OFFSET): (mod_short, mod_full)
    for (contig, bio_pos), (mod_short, mod_full) in KNOWN_MODIFICATIONS.items()
}
KNOWN_MOD_SET = set(KNOWN_MOD_PIPELINE.keys())

mod_counts = Counter(v[0] for v in KNOWN_MODIFICATIONS.values())
print(f'Total known modification sites: {len(KNOWN_MODIFICATIONS)}')
print(f'Modification types: {len(mod_counts)}')
print()
for mod_type, count in mod_counts.most_common():
    full = next(v[1] for v in KNOWN_MODIFICATIONS.values() if v[0] == mod_type)
    print(f'  {mod_type:8s}  {count} sites   ({full})')

## 4. Load Baleen Pipeline Results and Run Hierarchical Stages

In [ ]:
contig_results = None
for candidate in [BALEEN_PKL,
                  Path('../results/pipeline_results.pkl'),
                  Path('../output/pipeline_results.pkl'),
                  Path('output/pipeline_results.pkl')]:
    if candidate.exists():
        print(f'Loading: {candidate}')
        contig_results, metadata = load_results(candidate)
        break

if contig_results is None:
    raise FileNotFoundError(
        'pipeline_results.pkl not found.\n'
        'Run: baleen run --native-bam ... --ivt-bam ... --ref ... -o output/'
    )

print(f'Loaded {len(contig_results)} contigs')
for contig, cr in contig_results.items():
    known = sum(1 for p in cr.positions if (contig, p) in KNOWN_MOD_SET)
    print(f'  {contig}: {len(cr.positions):,} positions  ({known} known mod sites)')

In [ ]:
print('Running hierarchical pipeline (V1 → V3)...')
hier_results = {}
for contig, cr in contig_results.items():
    hier_results[contig] = compute_sequential_modification_probabilities(cr)
    print(f'  {contig}: done')
print('Done.')

## 5. Extract Baleen Site-Level Scores

In [ ]:
def extract_site_scores(contig_results, hier_results, known_mod_set, known_mod_pipeline):
    """Build per-position DataFrame with all baleen stage scores."""
    records = []
    for contig, cr in contig_results.items():
        cmr = hier_results.get(contig)
        for pos, pr in cr.positions.items():
            is_mod = (contig, pos) in known_mod_set
            mod_info = known_mod_pipeline.get((contig, pos), ('unmod', 'unmodified'))
            ps = cmr.position_stats.get(pos) if cmr else None

            n_nat = pr.n_native_reads
            n_ivt = pr.n_ivt_reads

            rec = {
                'contig':    contig,
                'position':  pos,
                'kmer':      pr.reference_kmer,
                'y_true':    int(is_mod),
                'mod_type':  mod_info[0],
                'n_native':  n_nat,
                'n_ivt':     n_ivt,
            }

            if ps is not None:
                native_slice = slice(0, ps.n_native)
                # V1: mean native z-score
                rec['v1_z']        = float(np.mean(ps.z_scores[native_slice]))
                # V2 raw: mean native mixture posterior
                rec['v2_raw']      = float(np.nanmean(ps.p_mod_raw[native_slice]))
                # V2 kNN: mean native kNN IVT-purity
                rec['v2_knn']      = float(np.nanmean(ps.p_mod_knn[native_slice]))
                # V3 HMM: mean native HMM-smoothed
                rec['v3_hmm']      = float(np.nanmean(ps.p_mod_hmm[native_slice]))
                # Stoichiometry (fraction of native reads with p_mod_hmm > 0.5)
                hmm_nat = ps.p_mod_hmm[native_slice]
                hmm_valid = hmm_nat[~np.isnan(hmm_nat)]
                rec['stoichiometry'] = float(np.mean(hmm_valid > 0.5)) if len(hmm_valid) > 0 else np.nan
                # Gate weight (soft gate for V2 raw)
                rec['gate_weight'] = float(ps.gate_weight)
                # Coverage class
                rec['coverage_class'] = str(ps.coverage_class.value)
            else:
                for col in ['v1_z','v2_raw','v2_knn','v3_hmm','stoichiometry','gate_weight']:
                    rec[col] = np.nan
                rec['coverage_class'] = 'unknown'

            records.append(rec)

    return pd.DataFrame(records)


df_baleen = extract_site_scores(contig_results, hier_results, KNOWN_MOD_SET, KNOWN_MOD_PIPELINE)
print(f'Site records: {len(df_baleen):,}')
print(f'  Known modification sites: {df_baleen.y_true.sum():,}')
print(f'  Unmodified sites:         {(df_baleen.y_true == 0).sum():,}')
df_baleen.head(3)

## 5b. Load Baleen site_results.tsv (if available)

The `site_results.tsv` produced by `baleen run` contains the final aggregated scores including `pvalue`, `padj`, `mean_p_mod`, `stoichiometry`. Merge these in so we can compute `-log10(padj)` as a benchmarking score.

In [ ]:
df_site_tsv = None
for candidate in [BALEEN_SITES,
                  Path('../output/site_results.tsv'),
                  Path('../results/site_results.tsv')]:
    if candidate is not None and candidate.exists():
        df_site_tsv = pd.read_csv(candidate, sep='\t')
        print(f'Loaded site_results.tsv: {len(df_site_tsv):,} rows')
        print(f'Columns: {list(df_site_tsv.columns)}')
        break

if df_site_tsv is not None:
    # Merge padj and pvalue back
    merge_cols = ['contig', 'position']
    extra = [c for c in ['pvalue', 'padj', 'mean_p_mod', 'median_p_mod']
             if c in df_site_tsv.columns]
    df_baleen = df_baleen.merge(
        df_site_tsv[merge_cols + extra],
        on=merge_cols, how='left'
    )
    print(f'Merged extra columns: {extra}')
else:
    print('site_results.tsv not found — regenerating from in-memory results.')
    # Regenerate site TSV from hierarchical results
    sites = aggregate_all(hier_results)
    df_site_tsv = pd.DataFrame([{
        'contig': s.contig, 'position': s.position, 'kmer': s.kmer,
        'n_native': s.n_native, 'n_ivt': s.n_ivt,
        'mean_p_mod': s.mean_p_mod, 'median_p_mod': s.median_p_mod,
        'stoichiometry': s.stoichiometry,
        'pvalue': s.pvalue, 'padj': s.padj,
    } for s in sites])
    df_baleen = df_baleen.merge(
        df_site_tsv[['contig','position','pvalue','padj']],
        on=['contig','position'], how='left'
    )
    print(f'Generated {len(df_site_tsv):,} site rows')

## 6. Compute Derived Scores

In [ ]:
df = df_baleen.copy()

# -log10(padj) as a continuous score for ROC/PR evaluation
# Using -log10 rather than 1-padj because it spreads significant values
# over a wider range (padj=0.001→3.0, padj=0.01→2.0) instead of
# compressing them near 1.0 (padj=0.001→0.999, padj=0.01→0.99).
if 'padj' in df.columns:
    padj_clipped = df['padj'].fillna(1.0).clip(lower=1e-300)
    df['neg_log10_padj'] = -np.log10(padj_clipped)

print(f'Site DataFrame: {len(df):,} rows x {len(df.columns)} columns')
print(f'Columns: {list(df.columns)}')

## 7. Define Methods Table

In [ ]:
# (column, display_label, color, group)
ALL_METHODS = [
    ('v1_z',            'V1 z-score',       '#CE93D8', 'baleen-v1'),
    ('v2_raw',          'V2 raw',           '#80CBC4', 'baleen-v2'),
    ('v2_knn',          'V2 kNN',           '#29B6F6', 'baleen-v2'),
    ('v3_hmm',          'V3 HMM',           '#EF5350', 'baleen-v3'),
    ('stoichiometry',   'Stoichiometry',    '#FF7043', 'baleen-v3'),
    ('neg_log10_padj',  '-log10(padj)',     '#EC407A', 'baleen-stat'),
]

available_methods = [
    (col, label, color, group)
    for col, label, color, group in ALL_METHODS
    if col in df.columns and df[col].notna().sum() > 3
]

y_true = df['y_true'].values

print(f'Available methods: {len(available_methods)}')
for col, label, color, group in available_methods:
    n = df[col].notna().sum()
    print(f'  [{group:12s}] {label:18s}  non-NaN sites: {n:,}')

## 8. ROC and PR Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
ax_roc, ax_pr = axes

roc_summary = []
for col, label, color, group in available_methods:
    scores = df[col].values
    valid  = ~np.isnan(scores)
    if len(np.unique(y_true[valid])) < 2:
        continue

    fpr, tpr, _ = roc_curve(y_true[valid], scores[valid])
    roc_a = auc(fpr, tpr)
    prec, rec, _ = precision_recall_curve(y_true[valid], scores[valid])
    pr_a  = average_precision_score(y_true[valid], scores[valid])

    ax_roc.plot(fpr, tpr, '-', color=color, lw=2,
                label=f'{label} (AUC={roc_a:.3f})')
    ax_pr.plot(rec, prec, '-', color=color, lw=2,
               label=f'{label} (AP={pr_a:.3f})')

    roc_summary.append(dict(method=label, group=group,
                            roc_auc=roc_a, pr_auc=pr_a,
                            n_sites=int(valid.sum())))

ax_roc.plot([0,1],[0,1],'k--',lw=1,label='Chance')
baserate = y_true.mean()
ax_pr.axhline(baserate, color='k', ls='--', lw=1,
              label=f'Chance ({baserate:.3f})')

for ax, xl, yl, title in [
    (ax_roc, 'False Positive Rate', 'True Positive Rate',
     'Site-Level ROC Curves'),
    (ax_pr,  'Recall', 'Precision',
     'Site-Level Precision-Recall Curves'),
]:
    ax.set_xlabel(xl); ax.set_ylabel(yl); ax.set_title(title)
    ax.legend(fontsize=8, loc='best')
    ax.set_xlim([-0.02, 1.02]); ax.set_ylim([-0.02, 1.02])

plt.tight_layout()
plt.savefig(OUT_DIR / 'site_roc_pr.pdf', bbox_inches='tight')
plt.show()

sum_df = pd.DataFrame(roc_summary).sort_values('roc_auc', ascending=False)
print(sum_df.round(4).to_string(index=False))

## 9. AUC Bar Chart

In [ ]:
if len(roc_summary) == 0:
    print('No summary — run Section 8 first.')
else:
    sdf = pd.DataFrame(roc_summary).sort_values('roc_auc', ascending=True)
    group_colors = {
        'baleen-v1':   '#CE93D8',
        'baleen-v2':   '#29B6F6',
        'baleen-v3':   '#EF5350',
        'baleen-stat': '#EC407A',
    }
    fig, axes = plt.subplots(1, 2, figsize=(13, max(4, len(sdf)*0.45)))
    for ax, metric, title in [(axes[0],'roc_auc','ROC-AUC'),
                               (axes[1],'pr_auc', 'Average Precision (PR-AUC)')]:
        bar_colors = [group_colors.get(g, 'gray') for g in sdf['group']]
        bars = ax.barh(sdf['method'], sdf[metric], color=bar_colors, edgecolor='white')
        ax.axvline(0.5 if metric == 'roc_auc' else y_true.mean(),
                   color='gray', ls='--', lw=1)
        ax.set_xlim([0, 1.06])
        for bar, val in zip(bars, sdf[metric]):
            ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
                    f'{val:.3f}', va='center', fontsize=9)
        ax.set_xlabel(title); ax.set_title(title)

    from matplotlib.patches import Patch
    handles = [Patch(facecolor=c, label=g) for g, c in group_colors.items()
               if g in sdf['group'].values]
    axes[0].legend(handles=handles, title='Group', fontsize=8)
    plt.suptitle('Site-Level AUC: Baleen Pipeline Stages', fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'site_auc_bar.pdf', bbox_inches='tight')
    plt.show()

## 10. Probability Tracks Along Each Transcript

Plot per-position modification scores (V2 kNN, V3 HMM, and stoichiometry) along the transcript, with known modification sites highlighted in red.

In [ ]:
for contig in df['contig'].unique():
    sub = df[df['contig'] == contig].sort_values('position')
    if len(sub) < 5:
        continue

    fig, axes = plt.subplots(2, 1, figsize=(16, 7), sharex=True)

    positions   = sub['position'].values
    known_mask  = sub['y_true'].values.astype(bool)

    # ── Panel 1: Modification probability tracks ──
    ax = axes[0]
    ax.axhline(0.5, color='gray', ls=':', lw=1)
    for col, label, color in [
        ('v2_knn', 'V2 kNN',  '#29B6F6'),
        ('v3_hmm', 'V3 HMM',  '#EF5350'),
        ('stoichiometry', 'Stoichiometry', '#FF7043'),
    ]:
        if col in sub.columns:
            ax.plot(positions, sub[col].values, 'o-', color=color,
                    ms=3, lw=1.2, alpha=0.8, label=label)
    for p in positions[known_mask]:
        ax.axvspan(p - 1, p + 1, alpha=0.25, color='red')
    ax.set_ylabel('P(modified)')
    ax.set_ylim([-0.05, 1.05])
    ax.set_title(f'{contig}  —  Modification Probability Tracks')
    ax.legend(fontsize=9, loc='upper right')

    # ── Panel 2: V1 z-scores ──
    ax = axes[1]
    bar_colors = np.where(known_mask, '#EF5350', '#90A4AE')
    ax.bar(positions, sub['v1_z'].values, color=bar_colors, width=0.8, alpha=0.8)
    ax.axhline(2.0,  color='orange', ls='--', lw=1, label='z = 2')
    ax.axhline(-2.0, color='orange', ls='--', lw=1)
    ax.axhline(0,    color='k',      ls='-',  lw=0.5)
    ax.set_ylabel('Mean V1 z-score')
    ax.set_xlabel('Genomic Position (0-based)')
    ax.set_title('V1 Z-Scores (red bars = known modified)')
    ax.legend(fontsize=9)

    plt.tight_layout()
    plt.savefig(OUT_DIR / f'transcript_tracks_{contig}.pdf', bbox_inches='tight')
    plt.show()

## 11. Performance by Modification Type (Heatmap)

In [ ]:
mod_types = [mt for mt in df['mod_type'].unique() if mt != 'unmod']

mod_auc_records = []
for mt in mod_types:
    y_ovr = (df['mod_type'] == mt).astype(int).values
    if y_ovr.sum() == 0:
        continue
    for col, label, color, group in available_methods:
        scores = df[col].values
        valid  = ~np.isnan(scores)
        if len(np.unique(y_ovr[valid])) < 2:
            continue
        try:
            roc_a = auc(*roc_curve(y_ovr[valid], scores[valid])[:2])
            pr_a  = average_precision_score(y_ovr[valid], scores[valid])
        except Exception:
            roc_a, pr_a = np.nan, np.nan
        mod_auc_records.append(dict(
            mod_type=mt, method=label, group=group,
            roc_auc=roc_a, pr_auc=pr_a, n_sites=int(y_ovr.sum())
        ))

if mod_auc_records:
    mad = pd.DataFrame(mod_auc_records)

    for metric, title in [('roc_auc', 'ROC-AUC'), ('pr_auc', 'PR-AUC')]:
        pivot = mad.pivot(index='mod_type', columns='method', values=metric)
        pivot = pivot.loc[pivot.mean(axis=1).sort_values(ascending=False).index]

        fig, ax = plt.subplots(figsize=(max(10, len(pivot.columns)*1.2),
                                         max(5, len(pivot)*0.5)))
        sns.heatmap(pivot, annot=True, fmt='.2f', cmap='RdYlGn',
                    vmin=0.3, vmax=1.0, linewidths=0.5,
                    ax=ax, cbar_kws={'label': title})
        ax.set_title(f'Site-Level {title} by Modification Type', fontsize=12)
        ax.set_xlabel('')
        plt.xticks(rotation=35, ha='right')
        plt.tight_layout()
        plt.savefig(OUT_DIR / f'site_{metric}_heatmap_by_mod_type.pdf', bbox_inches='tight')
        plt.show()
else:
    print('Not enough per-modification-type data.')

## 12. Score Distributions at Known Modification Sites

Visualize how well each method separates known-modified from unmodified positions.

In [ ]:
n_methods = len(available_methods)
ncols = 3
nrows = (n_methods + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4, nrows * 3.2))
axes_flat = axes.flatten() if n_methods > 1 else [axes]

for ax, (col, label, color, group) in zip(axes_flat, available_methods):
    scores = df[col].values
    valid  = ~np.isnan(scores)
    mod_s   = scores[valid & (y_true == 1)]
    unmod_s = scores[valid & (y_true == 0)]

    ax.hist(unmod_s, bins=25, alpha=0.55, density=True,
            color='#90A4AE', label=f'Unmod (n={len(unmod_s)})')
    ax.hist(mod_s,   bins=max(5, len(mod_s)//2 + 1),
            alpha=0.70, density=True,
            color=color, label=f'Mod (n={len(mod_s)})')

    if len(mod_s) > 0 and len(unmod_s) > 0:
        stat, pval = stats.mannwhitneyu(mod_s, unmod_s, alternative='greater')
        ax.set_title(f'{label}\nMWU p={pval:.2e}', fontweight='bold')
    else:
        ax.set_title(f'{label}', fontweight='bold')

    ax.legend(fontsize=7)
    ax.set_xlabel('Score')
    ax.set_ylabel('Density')

for ax in axes_flat[n_methods:]:
    ax.set_visible(False)

plt.suptitle('Site-Level Score Distributions: Modified vs Unmodified', fontsize=13)
plt.tight_layout()
plt.savefig(OUT_DIR / 'site_score_distributions.pdf', bbox_inches='tight')
plt.show()

## 13. IVT Coverage Class Analysis

Site-level performance broken down by IVT coverage: `high` (≥20), `medium` (5–19), `low` (1–4).

In [ ]:
cov_records = []
for cc in df['coverage_class'].dropna().unique():
    mask = (df['coverage_class'] == cc).values
    y_cc = y_true[mask]
    if len(np.unique(y_cc)) < 2:
        continue
    for col, label, color, group in available_methods:
        scores = df[col].values[mask]
        valid  = ~np.isnan(scores)
        if len(np.unique(y_cc[valid])) < 2:
            continue
        try:
            roc_a = auc(*roc_curve(y_cc[valid], scores[valid])[:2])
        except Exception:
            roc_a = np.nan
        cov_records.append(dict(coverage=cc, method=label, roc_auc=roc_a,
                                n_pos=int(y_cc.sum()), n_total=int(mask.sum())))

if cov_records:
    cdf = pd.DataFrame(cov_records)
    order = ['high', 'medium', 'low', 'zero', 'unknown']
    cdf['coverage'] = pd.Categorical(cdf['coverage'],
                                      categories=[c for c in order if c in cdf['coverage'].unique()],
                                      ordered=True)
    g = sns.catplot(data=cdf, kind='bar', x='method', y='roc_auc',
                    hue='coverage', height=5, aspect=2.0, palette='coolwarm_r')
    g.set_xticklabels(rotation=35, ha='right')
    g.set_axis_labels('Method', 'ROC-AUC')
    g.ax.axhline(0.5, color='gray', ls='--', lw=1)
    g.ax.set_ylim([0, 1.05])
    g.fig.suptitle('Site-Level ROC-AUC by IVT Coverage Class', y=1.02)
    plt.savefig(OUT_DIR / 'site_auc_by_coverage_class.pdf', bbox_inches='tight')
    plt.show()

## 14. FDR Precision-Recall Analysis

For the baleen statistical test (`padj`), evaluate precision and recall at different FDR thresholds.

In [ ]:
if 'padj' in df.columns and df['padj'].notna().sum() > 0:
    fdr_thresholds = [0.01, 0.05, 0.10, 0.20, 0.50]
    fdr_records = []
    for fdr in fdr_thresholds:
        called = (df['padj'].fillna(1.0) < fdr).values
        tp = (called & (y_true == 1)).sum()
        fp = (called & (y_true == 0)).sum()
        fn = (~called & (y_true == 1)).sum()
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        fdr_records.append(dict(fdr_threshold=fdr, n_called=called.sum(),
                                precision=prec, recall=rec,
                                true_positive=tp, false_positive=fp))

    fdr_df = pd.DataFrame(fdr_records)
    print('FDR Analysis (Baleen BH-adjusted p-value):')
    print(fdr_df.round(4).to_string(index=False))

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].semilogx(fdr_df['fdr_threshold'], fdr_df['precision'], 'bo-', label='Precision')
    axes[0].semilogx(fdr_df['fdr_threshold'], fdr_df['recall'],    'rs-', label='Recall')
    axes[0].set_xlabel('FDR Threshold (padj)'); axes[0].set_ylabel('Value')
    axes[0].set_title('Precision & Recall vs FDR Threshold'); axes[0].legend()
    axes[0].set_ylim([0, 1.05])

    axes[1].semilogx(fdr_df['fdr_threshold'], fdr_df['n_called'], 'g^-')
    axes[1].set_xlabel('FDR Threshold'); axes[1].set_ylabel('# Sites Called')
    axes[1].set_title('Number of Sites Called')

    plt.tight_layout()
    plt.savefig(OUT_DIR / 'fdr_pr_analysis.pdf', bbox_inches='tight')
    plt.show()
else:
    print('padj not available — run Section 5b or re-run baleen aggregate.')

## 15. Calibration Curves (Site Level)

In [ ]:
prob_methods = [(col, label, color, group)
                for col, label, color, group in available_methods
                if col not in ('v1_z', 'neg_log10_padj')]

ncols = 3
nrows = (len(prob_methods) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3.8, nrows * 3.2))
axes_flat = axes.flatten() if len(prob_methods) > 1 else [axes]

for ax, (col, label, color, group) in zip(axes_flat, prob_methods):
    scores = df[col].values
    valid  = ~np.isnan(scores)
    if valid.sum() < 20 or len(np.unique(y_true[valid])) < 2:
        ax.set_title(f'{label}\n(insufficient data)', color='gray')
        continue
    try:
        prob_true, prob_pred = calibration_curve(
            y_true[valid], scores[valid], n_bins=8, strategy='quantile'
        )
        ax.plot(prob_pred, prob_true, 'o-', color=color, lw=2, ms=5)
    except Exception as e:
        ax.text(0.5, 0.5, str(e)[:40], ha='center', va='center',
                transform=ax.transAxes, fontsize=7, color='red')
    ax.plot([0,1],[0,1],'k--',lw=1)
    ax.set_title(f'{label}', fontweight='bold')
    ax.set_xlabel('Mean predicted')
    ax.set_ylabel('Fraction positive')
    ax.set_xlim([-0.02, 1.02]); ax.set_ylim([-0.02, 1.02])

for ax in axes_flat[len(prob_methods):]:
    ax.set_visible(False)

plt.suptitle('Calibration Curves (Site Level)', fontsize=13)
plt.tight_layout()
plt.savefig(OUT_DIR / 'site_calibration.pdf', bbox_inches='tight')
plt.show()

## 16. Known Modification Site Scorecard

For each known modification site found in the data, show all method scores.

In [ ]:
known_df = df[df['y_true'] == 1].copy()

if len(known_df) > 0:
    score_cols = [col for col, *_ in available_methods if col in known_df.columns]
    display_cols = ['contig', 'position', 'kmer', 'mod_type',
                    'n_native', 'n_ivt', 'coverage_class'] + score_cols
    display_cols = [c for c in display_cols if c in known_df.columns]

    print(f'Known modification sites found in data: {len(known_df)}')
    print('='*90)
    with pd.option_context('display.max_columns', None, 'display.width', 200):
        print(known_df[display_cols].sort_values(['contig','position'])
              .round(3).to_string(index=False))

    # Summary: mean score per modification type per method
    print('\n── Mean scores by modification type ──────────────────')
    agg = known_df.groupby('mod_type')[score_cols].mean().round(3)
    agg['n_sites'] = known_df.groupby('mod_type').size()
    print(agg.to_string())
else:
    print('No known modification sites found in the output.')

## 17. Summary Table and Export

In [ ]:
summary_records = []
for col, label, color, group in available_methods:
    scores = df[col].values
    valid  = ~np.isnan(scores)
    if len(np.unique(y_true[valid])) < 2:
        continue

    fpr, tpr, _ = roc_curve(y_true[valid], scores[valid])
    roc_a = auc(fpr, tpr)
    pr_a  = average_precision_score(y_true[valid], scores[valid])

    prec_, rec_, t_ = precision_recall_curve(y_true[valid], scores[valid])
    f1_ = 2 * prec_ * rec_ / np.maximum(prec_ + rec_, 1e-9)
    best_f1 = f1_.max()
    best_t  = t_[f1_[:-1].argmax()] if len(t_) > 0 else np.nan

    summary_records.append(dict(
        Group=group, Method=label,
        ROC_AUC=roc_a, PR_AUC=pr_a,
        Best_F1=best_f1, Best_Threshold=best_t,
        N_sites=int(valid.sum()),
    ))

sum_df = pd.DataFrame(summary_records).sort_values(['Group','ROC_AUC'],
                                                     ascending=[True, False])
print('\n' + '='*80)
print('SITE-LEVEL BENCHMARK SUMMARY')
print('='*80)
print(f'Total sites: {len(df):,}')
print(f'Known modification sites: {y_true.sum():,}')
print(f'Unmodified sites: {(y_true==0).sum():,}')
print(f'Contigs: {df.contig.nunique()}')
print(f'Modification types in data: {df[df.y_true==1].mod_type.nunique()}')
print()
print(sum_df.round(4).to_string(index=False))

sum_df.to_csv(OUT_DIR / 'site_level_summary.csv', index=False)
df.to_csv(OUT_DIR / 'site_level_all_scores.csv', index=False)
print(f'\nSaved to {OUT_DIR}/')